# **Leitura inicial do arquivo dos votos gerados através do BU físico**

In [ ]:
import pandas as pd

# df = pd.read_csv(r'D:\Downloads\eleicoes_2026\tse_2026_rj\csv\votos_2026_rj.csv')
# df['codigo_votavel'] = df['codigo_votavel'].astype('Int32')

df = pd.read_parquet('https://huggingface.co/datasets/HugoCDM/tse_dados_2026_rj/resolve/main/votos_dados_2026_rj.parquet')

C:\Users\hugop\AppData\Local\Temp\ipykernel_24916\3947793006.py:3: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'D:\Downloads\eleicoes_2026\tse_2026_rj\csv\votos_2026_rj.csv')


# **Merge dados ea16 e ea18 (votos_2026_rj) com eleitorado_local_votacao_rj**
Os dados que vieram da extração da ea16, ea18 e bu físico+ local do eleitorado, que tem seção, zona, localização e geolocalização
- **ea16**: seções eleitorais e como se organizam. UF -> Município -> Zona -> Seção
- **ea18**: arquivo/bu físico que corresponde a cada seção
- **Bu .dat**: quais foram os votos


In [ ]:
usecols=['DS_ENDERECO', 'NM_BAIRRO', 'NM_MUNICIPIO', 'NR_CEP', 'NR_LATITUDE',	'NR_LONGITUDE', 'NR_LOCAL_VOTACAO', 'NM_LOCAL_VOTACAO', 'NR_SECAO', 'NR_ZONA']
eleitorado_rj_2026 = pd.read_csv('eleitorado_local_votacao_2026_RJ.csv', encoding='latin1', sep=';')
eleitorado_rj_2026 = eleitorado_rj_2026.rename(columns={
    'NR_ZONA': 'zona',
    'NR_SECAO': 'secao',
    'NR_LOCAL_VOTACAO': 'local_votacao',
    'NM_BAIRRO': 'bairro',
    'NM_MUNICIPIO': 'municipio'
})
print(eleitorado_rj_2026.columns)
colunas = ['codigo_cargo', 'cargo', 'codigo_votavel', 'DS_ENDERECO', 'bairro', 'municipio_y', 'NR_CEP', 'NR_LATITUDE', 'NR_LONGITUDE', 'local_votacao', 'NM_LOCAL_VOTACAO', 'secao', 'zona', 'quantidade_votos', 'tipo_voto']
merged = pd.merge(df, eleitorado_rj_2026, on=['zona', 'secao', 'local_votacao'], how='left')
merged = merged[colunas].rename(columns={'municipio_y': 'municipio'})
merged = merged.astype({
    'codigo_cargo': 'Int32',
    'codigo_votavel': 'Int32'
})
df # 7122339
merged.columns # 7114137


# **Análise votos governador**

In [54]:
votos_nulos_governador = merged.query("tipo_voto == 'nulo' & cargo == 'Governador'")
votos_brancos_governador = merged.query("tipo_voto == 'branco' & cargo == 'Governador'")
votos_pedro_duarte = merged.query("tipo_voto == 'nominal' & cargo == 'Deputado Estadual' & codigo_votavel == 55300")


votos_totais_governador_geral = merged.query("cargo == 'Governador'")['quantidade_votos'].sum() 
votos_totais_governador_nao_nulos = merged.query("cargo == 'Governador' & tipo_voto == 'nominal'")['quantidade_votos'].sum()
votos_eduardo_paes = merged.query("tipo_voto == 'nominal' & cargo == 'Governador' & codigo_votavel == 55")


votos_douglas_ruas = merged.query("tipo_voto == 'nominal' & cargo == 'Governador' & codigo_votavel == 22")

print('Votos totais de governador geral:', votos_totais_governador_geral)
print('Votos totais de governador não nulos:', votos_totais_governador_nao_nulos, str((votos_totais_governador_nao_nulos/votos_totais_governador_geral * 100).round(2)) + "%")
print('Votos nulos para governador (aproximado):', votos_nulos_governador['quantidade_votos'].sum(), str((votos_nulos_governador['quantidade_votos'].sum()/votos_totais_governador_geral * 100).round(2)) + "%")
print('Votos em branco para governador (aproximado):', votos_brancos_governador['quantidade_votos'].sum(), str((votos_brancos_governador['quantidade_votos'].sum()/votos_totais_governador_geral * 100).round(2)) + "%")
# print('Votos Pedro Duarte:', votos_pedro_duarte['quantidade_votos'].sum())
print()
print('Votos Eduardo Paes:', votos_eduardo_paes['quantidade_votos'].sum(), str((votos_eduardo_paes['quantidade_votos'].sum()/votos_totais_governador_nao_nulos *100).round(2)) + "%")
print('Votos Douglas Ruas:', votos_douglas_ruas['quantidade_votos'].sum(), str((votos_douglas_ruas['quantidade_votos'].sum()/votos_totais_governador_nao_nulos *100).round(2)) + "%")




Votos totais de governador geral: 9845867
Votos totais de governador não nulos: 8669038 88.05%
Votos nulos para governador (aproximado): 675292 6.86%
Votos em branco para governador (aproximado): 501537 5.09%

Votos Eduardo Paes: 3706984 42.76%
Votos Douglas Ruas: 4271199 49.27%


# **Merge dos votos com seção e zona e geolocalizacao + informações candidatos**

In [33]:
df_candidatos_2026 = pd.read_csv('consulta_cand_2026_BRASIL.csv', encoding='latin-1', sep=';', usecols=['SG_UE', 'CD_CARGO', 'DS_CARGO', 'NR_CANDIDATO', 'NM_URNA_CANDIDATO', 'NR_PARTIDO', 'SG_PARTIDO'])
df_candidatos_2026 = df_candidatos_2026.rename(columns=
    {'CD_CARGO': 'codigo_cargo',
     'NR_CANDIDATO': 'codigo_votavel'}
)
df_candidatos_2026 = df_candidatos_2026.astype({
    'codigo_cargo': 'Int32',
    'codigo_votavel': 'Int32'
})

df_candidatos_2026 = df_candidatos_2026.drop_duplicates(
    subset=['codigo_cargo', 'codigo_votavel']
)
df_candidatos_2026 = df_candidatos_2026[
    df_candidatos_2026['SG_UE'].isin(['BR', 'RJ'])
]
# Registros que possuem candidato/votável
merged_com_candidato = merged[
    merged['codigo_votavel'].notna()
].copy()

# Registros sem candidato/votável
merged_sem_candidato = merged[
    merged['codigo_votavel'].isna()
].copy()

# Faz o merge apenas dos que possuem código votável
merged_2 = pd.merge(
    merged_com_candidato,
    df_candidatos_2026,
    on=['codigo_cargo', 'codigo_votavel'],
    how='left',
    indicator=True
)

merged_2 = pd.concat(
    [merged_2, merged_sem_candidato],
    ignore_index=True
)



# **Baixar votação completa (sem nome urna)**

In [ ]:
# Baixar novo df
merged_2.to_csv('Eleições 2026 completas.csv', index=False)

# **Gerar candidatos no Rio de Janeiro**

In [ ]:
votos_rj_municipio = merged_2[merged_2['municipio'] == 'RIO DE JANEIRO']

raf = votos_rj_municipio[(votos_rj_municipio['codigo_votavel'] == 5501) & (votos_rj_municipio['codigo_cargo'] == 6)].groupby('bairro')['quantidade_votos'].sum().reset_index().sort_values(by='quantidade_votos', ascending=False) # raf
marcio = votos_rj_municipio[(votos_rj_municipio['codigo_votavel'] == 5555)  & (votos_rj_municipio['codigo_cargo'] == 6)].groupby('bairro')['quantidade_votos'].sum().reset_index().sort_values(by='quantidade_votos', ascending=False) # marcio
michelle = votos_rj_municipio[(votos_rj_municipio['codigo_votavel'] == 5500)  & (votos_rj_municipio['codigo_cargo'] == 6)].groupby('bairro')['quantidade_votos'].sum().reset_index().sort_values(by='quantidade_votos', ascending=False) # michelle
julio = votos_rj_municipio[(votos_rj_municipio['codigo_votavel'] == 1111)  & (votos_rj_municipio['codigo_cargo'] == 6)].groupby('bairro')['quantidade_votos'].sum().reset_index().sort_values(by='quantidade_votos', ascending=False) # julio lopes

print('Votos para o Rafael Aloísio Freitas', raf)
print('Votos para o Márcio Ribeiro', marcio)
print('Votos para a Michelle da Corrente do Bem', michelle)
print('Votos para o Rafael Aloísio Freitas', julio)


with pd.ExcelWriter('votos_candidatos_rio.xlsx', engine='openpyxl') as writer:
    raf.to_excel(writer, sheet_name='Rafael Freitas', index=False)
    marcio.to_excel(writer, sheet_name='Marcio Ribeiro', index=False)
    michelle.to_excel(writer, sheet_name='Michelle', index=False)
    julio.to_excel(writer, sheet_name='Julio Lopes', index=False)